In [1]:
%cd ~/Documents/polarsteps  /polarsteps-album-generator/backend

[Errno 2] No such file or directory: '/home/itay/Documents/polarsteps /polarsteps-album-generator/backend'
/home/itay/Documents/polarsteps/polarsteps-album-generator/backend


In [2]:
from pathlib import Path
from app.logic.segments import build_segments, SegmentKind, clean_points
import folium

from app.models.polarsteps import PSLocations, PSPoint, PSTrip

In [3]:
def plot_gps_comparison(
        raw_points: list[PSPoint],
        cleaned_points: list[PSPoint],
):
    """
    Generates an interactive HTML map comparing raw and cleaned GPS trajectories.
    Allows toggling between 'Raw' and 'Cleaned' layers.
    """
    if not raw_points and not cleaned_points:
        print("No data to plot.")
        return

    # 1. Extract coordinates for plotting
    raw_coords = [(p.lat, p.lon) for p in raw_points]
    cleaned_coords = [(p.lat, p.lon) for p in cleaned_points]

    # 2. Determine the center of the map
    # Default to the first cleaned point, or raw if cleaned is empty
    center_point = cleaned_coords[len(cleaned_coords) // 2] if cleaned_coords else raw_coords[0]

    # 3. Initialize the map
    m = folium.Map(location=center_point, zoom_start=6, tiles="cartodbpositron")

    # 4. Create Feature Groups (Layers) for toggling
    raw_layer = folium.FeatureGroup(name="Raw Data (Red)", show=True)
    cleaned_layer = folium.FeatureGroup(name="Cleaned Data (Blue)", show=True)

    # 5. Add Raw Data (Red)
    if raw_coords:
        # Draw the path
        folium.PolyLine(
            locations=raw_coords,
            color='red',
            weight=2,
            opacity=0.6,
            dash_array='5, 5'  # Dashed line for raw data
        ).add_to(raw_layer)

        # Add small markers for individual raw points
        for lat, lon in raw_coords:
            folium.CircleMarker(
                location=(lat, lon),
                radius=2,
                color='red',
                fill=True,
                fill_color='red'
            ).add_to(raw_layer)

    # 6. Add Cleaned Data (Blue)
    if cleaned_coords:
        # Draw the solid path
        folium.PolyLine(
            locations=cleaned_coords,
            color='blue',
            weight=4,
            opacity=0.8
        ).add_to(cleaned_layer)

        # Add markers for individual cleaned points
        for lat, lon in cleaned_coords:
            folium.CircleMarker(
                location=(lat, lon),
                radius=3,
                color='blue',
                fill=True,
                fill_color='blue'
            ).add_to(cleaned_layer)

    # 7. Add layers to map and enable the layer control
    raw_layer.add_to(m)
    cleaned_layer.add_to(m)
    folium.LayerControl().add_to(m)

    return m

In [4]:
COLORS = {"other": "blue", "hike": "green", "flight": "black"}

RAW_DATA = sorted(PSLocations.model_validate_json(
    (
        Path("tests/test_data/trip/south-america-2024-2025_14232450/locations.json")
    ).read_bytes()
).locations)

steps = PSTrip.model_validate_json((
                                       Path("tests/test_data/trip/south-america-2024-2025_14232450/trip.json")
                                   ).read_bytes()).all_steps
step_points = [
    PSPoint(
        lat=step.location.lat, lon=step.location.lon, time=step.datetime.timestamp()
    )
    for idx, step in enumerate(steps) if idx in range(200, 220)
]

In [5]:
points = (
    [point for point in RAW_DATA + step_points if step_points[0].time <= point.time <= step_points[-1].time])
segments = list(build_segments(points))

In [6]:
# Calculate stats
total_original = len(points)
total_new = sum(len(s.latlons) for s in segments)
print(
    f"Points reduced from {total_original} to {total_new} ({(1 - total_new / total_original) * 100:.1f}%)"
)

# Center map on the first point
m = folium.Map(segments[len(segments) // 2].latlons[0], zoom_start=5, tiles="cartodbpositron")

flight_c = 0
hike_c = 0

# Draw segments
for seg in segments:
    folium.PolyLine(
        seg.latlons,
        color=COLORS[seg.kind],
        tooltip=f"{seg.kind.capitalize()} Segment ({len(seg.latlons)} pts)",
    ).add_to(m)

    # Add markers for start/end of flights
    if seg.kind == SegmentKind.flight:
        flight_c += 1
        folium.Marker(
            seg.latlons[0],
            popup="Flight Takeoff",
            icon=folium.Icon(color=COLORS[SegmentKind.flight]),
        ).add_to(m)
        folium.Marker(
            seg.latlons[-1],
            popup="Flight Landing",
            icon=folium.Icon(color=COLORS[SegmentKind.flight]),
        ).add_to(m)
    # Add markers for start/end of treks
    if seg.kind == SegmentKind.hike:
        hike_c += 1
        folium.Marker(
            seg.latlons[0],
            popup="Hike start",
            icon=folium.Icon(color=COLORS[SegmentKind.hike]),
        ).add_to(m)
        folium.Marker(
            seg.latlons[-1],
            popup="Hike end",
            icon=folium.Icon(color=COLORS[SegmentKind.hike]),
        ).add_to(m)

print(f"{flight_c=}, {hike_c=}")
m

Points reduced from 1451 to 1437 (1.0%)
flight_c=3, hike_c=1
